# Paper 3 Open Items Runner

Closes four remaining open items from the paper in one notebook:

## Item 4: LongMemEval Gate 1 Completion (fast subset)
Re-runs the Gate 1 oracle + refinement study on LongMemEval-S cleaned.
Uses a small conversation subset (`LONGMEM_LIMIT=6`) and strict turn cap
(`LONGMEM_MAX_TURNS=40`) to finish in ~30 min on a T4.

## Item 1: Learned Harm Predictor
Trains a 2-head MLP from the MSC oracle ablation damage CSV, then
evaluates `semantic_harm_keep_compress_drop` against heuristic baselines.
Uses the oracle CSV produced in the earlier Gate 1 MSC run (from Drive).

## Item 2: Memory-Object Level Compression
Evaluates `semantic_object_keep_compress_drop` (object-granularity KCD)
vs turn-granularity baselines on MSC valid.
Optionally trains an object-aware harm predictor and adds
`semantic_object_harm_keep_compress_drop` to the comparison.

## Item 3: Larger-Model Validation (3B probe)
Runs geometry_KCD + semantic_KCD on the hard stress set with `qwen25_3b`
to check whether the signal-comparison result holds at higher capacity.

---
**Drive persistence**: all results are written incrementally to Google Drive.
A disconnect mid-run loses only the current conversation — prior ones are safe.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR = "/content/rt-geometry-memory"

MODEL_15B  = "qwen25_15b"   # main model for items 4, 1, 2
MODEL_3B   = "qwen25_3b"    # item 3 larger-model probe
BUDGETS    = "0.20,0.35,0.50"

# ── Item 4: LongMemEval Gate 1 (fast subset) ──────────────────────────────────
# 6 conversations × 40 turns × 3 budgets ≈ 25-35 min on T4
# Increase LONGMEM_LIMIT to 10 for a fuller run (~50-60 min)
LONGMEM_LIMIT        = 6     # conversations to process
LONGMEM_MAX_TURNS    = 40    # truncate each conversation to first N turns
LONGMEM_STRIDE       = 6     # sample every Nth turn as target
LONGMEM_MAX_TARGETS  = 6     # max target turns per conversation

# ── Items 1 & 2: MSC oracle + predictor ──────────────────────────────────────
# These re-use the MSC oracle that was already run (loaded from Drive).
# If Drive doesn't have the oracle CSV, re-runs a small MSC oracle first.
MSC_LIMIT       = 16   # conversations (same as original Gate 1 run)
MSC_STRIDE      = 6
MSC_MAX_TARGETS = 6

# ── Item 3: 3B probe ─────────────────────────────────────────────────────────
ITEM3_RUN_NAME = "paper3_3b_signal_comparison_v1"

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(result.stdout.strip() or "No GPU — switch runtime in Runtime > Change runtime type")

In [ ]:
# ── Clone and install ─────────────────────────────────────────────────────────
import os
%cd /content
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!bash scripts/colab_setup.sh

In [ ]:
# ── Mount Google Drive and symlink results ────────────────────────────────────
# Results are written directly to Drive — safe across disconnects.
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
DRIVE_RESULTS = "/content/drive/MyDrive/rt_results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

LOCAL_RESULTS = f"{REPO_DIR}/results"
if not os.path.islink(LOCAL_RESULTS):
    if os.path.isdir(LOCAL_RESULTS):
        shutil.copytree(LOCAL_RESULTS, DRIVE_RESULTS, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_RESULTS)
    os.symlink(DRIVE_RESULTS, LOCAL_RESULTS)
    print(f"Results → Drive: {DRIVE_RESULTS}")
else:
    print(f"Symlink already set: {LOCAL_RESULTS} → {os.readlink(LOCAL_RESULTS)}")

In [ ]:
# ── Reconnect recovery: check what's already on Drive ────────────────────────
# Run this after a disconnect to see what's done before re-running any cells.
import os, glob

DRIVE_RESULTS = "/content/drive/MyDrive/rt_results"

checkpoints = {
    "Item4 LongMemEval oracle":    f"{DRIVE_RESULTS}/paper3/harm_oracle/paper3_gate1_oracle_longmemeval_s_cleaned_subset/oracle_summary.md",
    "Item4 LongMemEval refinement": f"{DRIVE_RESULTS}/paper3/studies/paper3_gate1_refinement_longmemeval_s_cleaned_subset/pairwise_report.md",
    "Item1 Harm predictor model":   f"{DRIVE_RESULTS}/paper3/harm_predictor_models/paper3_harm_predictor_msc_v1/harm_predictor.pt",
    "Item1 Predictor pairwise":     f"{DRIVE_RESULTS}/paper3/studies/paper3_harm_predictor_msc_v1/pairwise_report.md",
    "Item2 Object compression":     f"{DRIVE_RESULTS}/paper3/studies/paper3_semantic_object_msc_v1/pairwise_report.md",
    "Item3 3B probe pairwise":      f"{DRIVE_RESULTS}/paper3/studies/{ITEM3_RUN_NAME}/pairwise_report.md",
}

for label, path in checkpoints.items():
    status = "✓ done" if os.path.exists(path) else "✗ not yet run"
    print(f"  {status}  {label}")

---
## Download benchmarks from HuggingFace
Downloads MSC valid and LongMemEval-S cleaned. Skips if already present.

In [ ]:
%cd {REPO_DIR}

MSC_RAW      = f"{REPO_DIR}/benchmarks/msc_valid_raw.jsonl"
MSC_NORM     = f"{REPO_DIR}/benchmarks/msc_valid_normalized.jsonl"
LONGMEM_RAW  = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_raw.json"
LONGMEM_NORM = f"{REPO_DIR}/benchmarks/longmemeval_s_cleaned_normalized.jsonl"

if not __import__('os').path.exists(MSC_NORM):
    print("Downloading MSC valid...")
    !python scripts/download_public_benchmark.py --benchmark msc_valid --output "{MSC_RAW}"
    !python scripts/prepare_public_benchmark_jsonl.py --format msc --input "{MSC_RAW}" --output "{MSC_NORM}" --family msc_valid
    print(f"MSC valid ready: {MSC_NORM}")
else:
    print(f"MSC valid present: {MSC_NORM}")

if not __import__('os').path.exists(LONGMEM_NORM):
    print("Downloading LongMemEval-S cleaned...")
    !python scripts/download_public_benchmark.py --benchmark longmemeval_s_cleaned --output "{LONGMEM_RAW}"
    !python scripts/prepare_public_benchmark_jsonl.py --format longmemeval --input "{LONGMEM_RAW}" --output "{LONGMEM_NORM}" --family longmemeval_s_cleaned
    print(f"LongMemEval ready: {LONGMEM_NORM}")
else:
    print(f"LongMemEval present: {LONGMEM_NORM}")

import os
for label, path in [("MSC valid", MSC_NORM), ("LongMemEval-S", LONGMEM_NORM)]:
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    lines   = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    print(f"{label}: {lines} conversations, {size_mb:.1f} MB")

---
## Item 4: LongMemEval Gate 1 Completion (fast subset)

**What this runs:**
- Oracle headroom probe: how much Kendall tau could geometry gain inside the semantic shortlist?
- Refinement study: does `semantic_query_conditioned_geometry_KCD` beat `budget_aware_semantic_KCD`?

**Speed knobs (set in Config cell above):**
- `LONGMEM_LIMIT=6`, `LONGMEM_MAX_TURNS=40`, `LONGMEM_STRIDE=6` → ~30 min on T4
- Raise `LONGMEM_LIMIT` to 10 for a fuller run (~55 min)

In [ ]:
%cd {REPO_DIR}

# Oracle headroom
!bash scripts/run_paper3_harm_oracle_probe.sh \
    paper3_gate1_oracle_longmemeval_s_cleaned_subset \
    longmemeval_s_cleaned \
    "{LONGMEM_NORM}" \
    "{MODEL_15B}" \
    "{BUDGETS}" \
    {LONGMEM_LIMIT} \
    {LONGMEM_STRIDE} \
    {LONGMEM_MAX_TARGETS} \
    {LONGMEM_MAX_TURNS}

# Refinement study
!bash scripts/run_paper3_gate1_refinement_probe.sh \
    paper3_gate1_refinement_longmemeval_s_cleaned_subset \
    "{LONGMEM_NORM}" \
    "{MODEL_15B}" \
    "{BUDGETS}" \
    {LONGMEM_LIMIT} \
    {LONGMEM_STRIDE} \
    {LONGMEM_MAX_TARGETS}

In [ ]:
# ── Read Item 4 results ───────────────────────────────────────────────────────
import os, glob
%cd {REPO_DIR}

for oracle_name in ["paper3_gate1_oracle_longmemeval_s_cleaned_subset"]:
    path = f"results/paper3/harm_oracle/{oracle_name}/oracle_summary.md"
    if os.path.exists(path):
        print(f"\n{'='*60}\nOracle: {oracle_name}\n{'='*60}")
        with open(path) as f: print(f.read())
    else:
        for p in sorted(glob.glob(f"results/paper3/harm_oracle/{oracle_name}/*.md")):
            print(f"\n{'='*60}\n{p}\n{'='*60}")
            with open(p) as f: print(f.read())

for study_name in ["paper3_gate1_refinement_longmemeval_s_cleaned_subset"]:
    path = f"results/paper3/studies/{study_name}/pairwise_report.md"
    if os.path.exists(path):
        print(f"\n{'='*60}\nRefinement: {study_name}\n{'='*60}")
        with open(path) as f: print(f.read())
    else:
        print(f"Not found: {path}")

---
## Item 1: Learned Harm Predictor

Trains a 2-head MLP (logit damage + answer NLL damage) from the MSC oracle
ablation rows already on Drive, then runs `semantic_harm_keep_compress_drop`
against heuristic baselines.

**Oracle input:** `results/paper3/harm_oracle/paper3_gate1_oracle_msc_valid/candidate_rows.csv`  
If this CSV isn't on Drive (first run), the oracle is re-run on MSC first.

In [ ]:
# ── Ensure MSC oracle CSV exists (re-run if not on Drive) ─────────────────────
import os
%cd {REPO_DIR}

MSC_ORACLE_CSV = f"results/paper3/harm_oracle/paper3_gate1_oracle_msc_valid/candidate_rows.csv"

if not os.path.exists(MSC_ORACLE_CSV):
    print("MSC oracle CSV not found on Drive — running oracle probe first (~45 min)...")
    !bash scripts/run_paper3_harm_oracle_probe.sh \
        paper3_gate1_oracle_msc_valid \
        msc_valid \
        "{MSC_NORM}" \
        "{MODEL_15B}" \
        "{BUDGETS}" \
        {MSC_LIMIT} \
        {MSC_STRIDE} \
        {MSC_MAX_TARGETS}
else:
    import subprocess
    n = int(subprocess.check_output(["wc", "-l", MSC_ORACLE_CSV]).split()[0]) - 1
    print(f"MSC oracle CSV found on Drive: {n} candidate rows → ready for training")

In [ ]:
# ── Train harm predictor + evaluate ──────────────────────────────────────────
%cd {REPO_DIR}

!bash scripts/run_paper3_harm_predictor_probe.sh \
    paper3_harm_predictor_msc_v1 \
    "{MSC_NORM}" \
    "results/paper3/harm_oracle/paper3_gate1_oracle_msc_valid" \
    "{MODEL_15B}" \
    "{BUDGETS}" \
    {MSC_LIMIT} \
    {MSC_STRIDE} \
    {MSC_MAX_TARGETS}

In [ ]:
# ── Read Item 1 results ───────────────────────────────────────────────────────
import os, json
%cd {REPO_DIR}

# Predictor training summary
summary_path = "results/paper3/harm_predictor_models/paper3_harm_predictor_msc_v1/training_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print("Harm predictor training summary:")
    print(f"  Selected variant: {summary['selected_variant']}")
    print(f"  Attention Kendall delta: {summary['attention_validation_kendall_delta']:.4f}")
    for variant, info in summary.get('variants', {}).items():
        vm = info.get('validation_metrics', {}).get('row_level', {})
        print(f"  [{variant}] val Kendall={vm.get('kendall_tau', 'n/a'):.4f}  val Spearman={vm.get('spearman', 'n/a'):.4f}")

# Pairwise report
pairwise_path = "results/paper3/studies/paper3_harm_predictor_msc_v1/pairwise_report.md"
if os.path.exists(pairwise_path):
    print(f"\n{'='*60}\nPairwise report: paper3_harm_predictor_msc_v1\n{'='*60}")
    with open(pairwise_path) as f: print(f.read())
else:
    print(f"Not found: {pairwise_path}")

---
## Item 2: Memory-Object Level Compression

Evaluates whether compressing at the **semantic-object granularity** (persona
bundle, event bundle, constraint bundle) beats turn-granularity baselines.

Policies compared:
- `semantic` (baseline)
- `budget_aware_semantic_keep_compress_drop` (Gate 1 winner)
- `semantic_object_keep_compress_drop` ← object-level KCD
- `semantic_object_harm_keep_compress_drop` ← object-level + learned predictor

In [ ]:
%cd {REPO_DIR}

# Pass oracle dir so the script trains the object-aware predictor variant too
!bash scripts/run_paper3_semantic_object_probe.sh \
    paper3_semantic_object_msc_v1 \
    "{MSC_NORM}" \
    "results/paper3/harm_oracle/paper3_gate1_oracle_msc_valid" \
    "{MODEL_15B}" \
    "{BUDGETS}" \
    {MSC_LIMIT} \
    {MSC_STRIDE} \
    {MSC_MAX_TARGETS}

In [ ]:
# ── Read Item 2 results ───────────────────────────────────────────────────────
import os
%cd {REPO_DIR}

pairwise_path = "results/paper3/studies/paper3_semantic_object_msc_v1/pairwise_report.md"
if os.path.exists(pairwise_path):
    print(f"\n{'='*60}\nObject compression pairwise report\n{'='*60}")
    with open(pairwise_path) as f: print(f.read())
else:
    print(f"Not found: {pairwise_path}")

---
## Item 3: Larger-Model Validation (3B probe)

Reruns the Experiment 1 signal comparison (`geometry_KCD` vs `semantic_KCD`
on the hard stress set) with `qwen25_3b` instead of `qwen25_15b`.

Decision rule:
- If `geometry_KCD` still beats `semantic_KCD` at 3B → signal result is model-size robust
- If the gap shrinks or reverses → representational resolution matters for the signal

In [ ]:
%cd {REPO_DIR}

!bash scripts/run_paper3_signal_comparison_hardset.sh \
    "{ITEM3_RUN_NAME}" \
    "{MODEL_3B}" \
    "{BUDGETS}"

In [ ]:
# ── Read Item 3 results ───────────────────────────────────────────────────────
import os, glob
%cd {REPO_DIR}

pairwise_path = f"results/paper3/studies/{ITEM3_RUN_NAME}/pairwise_report.md"
if os.path.exists(pairwise_path):
    print(f"\n{'='*60}\n3B signal comparison pairwise report\n{'='*60}")
    with open(pairwise_path) as f: print(f.read())
else:
    print(f"Not found: {pairwise_path}")

# Memory critical analysis at budget 0.35
for pattern in [
    f"results/paper3/studies/{ITEM3_RUN_NAME}/memory_critical_*geometry_kcd_b035.md",
    f"results/paper3/studies/{ITEM3_RUN_NAME}/memory_critical_*semantic_kcd_b035.md",
]:
    for path in sorted(glob.glob(pattern)):
        print(f"\n{'='*60}\n{path}\n{'='*60}")
        with open(path) as f: print(f.read())

---
## Save and download all results

In [ ]:
%cd {REPO_DIR}

!zip -r /content/rt_open_items_results.zip \
    results/paper3/harm_oracle/paper3_gate1_oracle_longmemeval_s_cleaned_subset \
    results/paper3/studies/paper3_gate1_refinement_longmemeval_s_cleaned_subset \
    results/paper3/harm_predictor_models/paper3_harm_predictor_msc_v1 \
    results/paper3/studies/paper3_harm_predictor_msc_v1 \
    results/paper3/studies/paper3_semantic_object_msc_v1 \
    results/paper3/studies/{ITEM3_RUN_NAME} \
    2>/dev/null || echo "Some dirs not found — only completed runs were zipped."

from google.colab import files
files.download("/content/rt_open_items_results.zip")